# 🎯 [K09_M01] 자주 묻는 부대 생활 정보 파악하기
### 학생 실습 워크북 & 프롬프트 설계 가이드 (Google Colab / Jupyter)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/K09/k09_m01_student.ipynb)

---

## 📌 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 / 과제** | 병영생활·상담 (Level 2) / K09 부대 정보 검색 시스템 |
| **미션** | 미션 01. 자주 묻는 부대 생활 정보 파악하기 (소요시간: 약 1.5시간) |
| **실습 유형** | 생성형 AI 프롬프트 엔지니어링 및 팩트체크 검증 |

> **⚠️ [교육용 가상 자료 원칙]**
> 본 실습의 모든 자료는 안전 규정에 따라 **교육용 가상 부대 자료(가상 한빛대대)**로 구성되어 있습니다.
> 실제 군사 기밀이나 부대 세부 인적·작전 정보는 외부 AI 도구에 절대 입력하지 않습니다.



In [ ]:
# ============================================================================
# [환경 설정 및 헬퍼 함수] 마크다운 시각화 렌더러 및 가상 데이터셋 로더
# ============================================================================
from IPython.display import display, Markdown
import json
import os

def load_dataset(mission_num):
    """미션 번호(1~10)에 해당하는 가상 데이터셋 JSON을 로드하여 반환합니다."""
    filename = f"k09_m{mission_num:02d}_dataset.json"
    for base in [".", "uploads/K09", "../uploads/K09"]:
        filepath = os.path.join(base, filename)
        if os.path.exists(filepath):
            with open(filepath, "r", encoding="utf-8") as f:
                return json.load(f)
    print(f"⚠️ 데이터셋 파일을 찾을 수 없습니다: {filename}")
    return None

def show_dataset_summary(mission_num):
    """해당 미션의 데이터셋 주요 항목을 요약 출력합니다."""
    data = load_dataset(mission_num)
    if data:
        display(Markdown(f"**[데이터셋 로드 성공]** `{data.get('title', '')}` ({data.get('mission', '')})"))
        keys = [k for k in data.keys() if k not in ["project", "mission", "title"]]
        print(f"포함 항목: {', '.join(keys)}")

def show_result(mission_title, prompt, result):
    """학습자가 입력한 프롬프트와 AI 답변 및 검증 결과를 시각적으로 렌더링하는 헬퍼 함수"""
    p_text = prompt.strip() if prompt.strip() else "(작성된 프롬프트가 없습니다)"
    r_text = result.strip() if result.strip() else "(기록된 결과가 없습니다)"
    
    md_content = f"""
---
### 📌 [{mission_title}] 실습 기록 리포트

#### 💬 입력 프롬프트
```text
{p_text}
```

#### 📋 AI 생성 결과 및 팩트체크 검증 내역
{r_text}
---
"""
    display(Markdown(md_content))

print("✅ 워크북 시각화 헬퍼 함수 및 가상 데이터셋 로더(k09_m01~m10) 준비 완료!")



---

## 🎯 미션 01. 자주 묻는 부대 생활 정보 파악하기
**소요시간**: 약 1.5시간

![미션 1 부대 생활 질문 분류 및 신병 우선순위 분석 다이어그램](/uploads/common/K09_m01_question_taxonomy.png)

### 📋 미션 개요 및 배경
신규 전입 장병이 부대 적응 과정에서 겪는 정보 탐색의 비효율과 반복적인 구두 질문으로 인한 행정 소요를 분석합니다. 실습자료의 질문 카드 20개를 시설·시간·절차·담당 부서 기준으로 분류하고, 전입 첫 주에 반드시 필요한 핵심 질문 5개를 선별합니다. 기존 정보 탐색 방식의 한계를 진단하여 생성형 AI 기반 안내 도우미가 해결해야 할 핵심 과업을 명확히 정의합니다.

### 🎯 미션 학습목표
실습자료의 질문 카드 20개를 시설·시간·절차·담당 부서로 분류하고, 전입 첫 주에 가장 필요한 핵심 질문 5개를 선별하여 현재 정보 탐색 방식의 불편점을 분석합니다. **이를 통해 사용자 관점에서 현장 문제를 정의하고, AI 도입이 필요한 우선순위 요구사항을 분석·선정하는 문제 도출 역량을 익힙니다.**

### 📑 실습 가이드 및 세부 단계 (프롬프트 & 검증 포인트)
#### 🔹 Step 1. 질문 카드 20개를 시설·시간·절차·담당 부서로 분류한다.
#### 이 단계의 역할
가상 한빛대대 장병들의 부대 생활 질문 카드 20개를 시설 위치, 운영 시간, 신청 절차, 담당 부서의 4대 카테고리로 체계적으로 분류합니다.

#### 해야 할 일

<li>실습 데이터(JSON)에 수록된 20개의 가상 질문 카드 원문을 꼼꼼히 확인합니다.

</li><li>각 질문의 핵심 키워드를 파악하여 [시설 위치, 운영 시간, 신청 절차, 담당 부서] 카테고리 분류표에 배치합니다.

</li><li>AI에게 복합적이거나 분류가 애매한 질문에 대한 최적의 카테고리 분류 기준을 자문하고 조율합니다.

</li>

#### 💡 AI 프롬프트 질의 예시
<blockquote>**[질문 카테고리 분류 질의]:** "너는 군 행정 및 병영생활 상담 전문가야. 다음 장병들의 질문 20개를 [시설 위치, 운영 시간, 신청 절차, 담당 부서] 4개 카테고리로 분류하고, 분류 기준이 모호한 질문은 그 판단 근거를 간단히 적어줘.
[질문 카드 목록]: {질문 카드 20개 원문}"

</blockquote>
#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **누락 없는 전수 분류:** 20개 질문 카드가 중복이나 누락 없이 모두 분류표에 포함되었는가?
- **카테고리 타당성:** 단순 위치 문의(예: 사이버지식정보방 위치)와 운영 시간 및 행정 절차가 명확히 구분되었는가?

#### 핵심 확인 지표
<blockquote>20개 질문 카드 4대 영역 분류 완료표

</blockquote>
#### 학습자가 제출해야 하는 내용

<li>질문 카드 20개 분류표 완성본

</li>

#### 🔹 Step 2. 전입 첫 주 우선 질문 5개를 고른다.
#### 이 단계의 역할
20개 질문 중 신규 전입 장병이 자대 배치 첫 주 적응 과정에서 가장 시급하게 필요로 하는 우선순위 Top 5 질문을 선별합니다.

#### 해야 할 일

<li>전입 1~7일 차 장병의 필수 생활 동선(생활관, 식당, 의무실, PX, 복지시설)을 시뮬레이션합니다.

</li><li>질문 카드 20개 중 미확인 시 생활 불편이나 규정 위반이 발생할 수 있는 5개 핵심 질문을 선정합니다.

</li><li>AI에게 신병 관점에서의 질문 시급도 및 중요도 평가를 요청하고 본인의 선별안과 비교 검토합니다.

</li>

#### 💡 AI 프롬프트 질의 예시
<blockquote>**[신병 우선 질문 선별 질의]:** "신병이 자대에 전입 온 첫 주에 겪는 주요 애로사항을 고려할 때, 다음 20개 질문 중 가장 시급하고 중요한 우선 질문 5개를 추천하고 선별 이유를 설명해 줘.
[질문 목록]: {분류된 20개 질문}"

</blockquote>
#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **우선순위 적절성:** 의무실 이용 시간, 식사 시간, 필수 행정 절차 등 긴급도 높은 질문이 포함되었는가?
- **선별 사유 명확성:** 각 질문이 전입 첫 주에 왜 우선되어야 하는지 타당한 근거가 제시되었는가?

#### 핵심 확인 지표
<blockquote>전입 첫 주 우선 질문 5개 및 선정 사유서

</blockquote>
#### 학습자가 제출해야 하는 내용

<li>우선 질문 5개 목록 및 선정 근거서

</li>

#### 🔹 Step 3. 현재 정보 탐색 방식과 불편한 점을 적는다.
#### 이 단계의 역할
선임병 구두 질의, 게시판 공지 확인 등 기존 정보 탐색 경로의 한계와 병영 내 소통 단절 문제를 구체적으로 분석합니다.

#### 해야 할 일

<li>선임병 질문, 행정반 방문, 게시판 육안 확인 등 기존 방식의 소요 시간과 심리적 부담을 정리합니다.

</li><li>구두 전파 과정에서의 정보 왜곡, 담당자 부재 시 확인 지연 등 구체적인 병목 현상을 3가지 이상 도출합니다.

</li><li>AI 기반 부대 생활 안내 도우미 도입 시 기대되는 개선 효과(시간 절약, 익명성, 정확도)를 대조표로 작성합니다.

</li>

#### 💡 AI 프롬프트 질의 예시
<blockquote>**[기존 탐색 방식 한계 분석 질의]:** "군 부대에서 신병이 선임병이나 간부에게 매번 구두로 질문할 때 겪는 심리적 부담과 정보 왜곡 문제를 분석해 줘. 이를 생성형 AI 도우미로 대체했을 때 기대되는 실질적인 업무/생활 개선 효과 3가지를 도출해 줘."

</blockquote>
#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **현장 애로사항 반영:** 군 생활 현장에서 실제로 발생하는 현실적인 소통 불편 사항이 구체적으로 기술되었는가?
- **AI 도입 당위성 연결:** 기존 방식의 문제점이 향후 AI 안내 도우미 구축 목적과 논리적으로 직결되는가?

#### 핵심 확인 지표
<blockquote>정보 탐색 방식 비교 분석표 및 문제 진단서

</blockquote>
#### 학습자가 제출해야 하는 내용

<li>현재 정보 탐색 방식과 불편점 정리서

</li>

#### 🔹 Step 4. 심화 도전 과제 및 최적화
#### 이 단계의 역할
기본 실습 완료 후 실제 자대 환경을 반영한 추가 질문 2건을 직접 고안하여 분류 체계를 확장합니다.

#### 해야 할 일
실제 전입 시 가장 궁금했던 현실적 질문 2개를 직접 추가하고 카테고리를 분류해 본다.

#### 💡 AI 프롬프트 질의 예시
<blockquote>**[자대 맞춤형 심화 질문 개발 질의]:** "부대 생활 안내 시스템에 다음 2개 질문을 추가하려 해. [질문 1: 주말 생활관 세탁기/건조기 이용 순번 규칙, 질문 2: 전투휴무일 종교시설 및 복지시설 운영 시간]. 이 질문들이 신병 적응에 기여하는 바와 적합한 카테고리를 평가해 줘."

</blockquote>
#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **군사보안 준수:** 실제 부대 고유 명칭, 작전 계획, 간부 개인정보 등 비인가 정보가 일체 포함되지 않았는가?
- **질문 실효성:** 추가된 질문이 병영 생활 편의와 규정 준수에 실질적으로 기여하는가?

#### 핵심 확인 지표
<blockquote>심화 질문 2건 추가 및 타당성 분석 메모

</blockquote>
#### 학습자가 제출해야 하는 내용

<li>심화 도전 과제 수행 결과 및 분석 메모

</li>

### 📝 실습 서식 및 모범 작성 양식 (참고 템플릿)
# [K09_M01] 자주 묻는 부대 생활 정보 파악하기 실습 가이드 및 작성 서식

## 1. 부대 생활 질문 20개 분류표
- 질문 카드 20건을 시설 위치, 운영 시간, 신청 절차, 담당 부서 4대 영역으로 분류합니다.

| 카테고리 | 질문 번호 (ID) | 질문 내용 요약 | 분류 사유 |
| :--- | :--- | :--- | :--- |
| **시설 위치** | Q07, Q16, ... | 영내 분실물 보관소 및 흡연장 위치 문의 | 시설의 공간적 위치 확인 목적 |
| **운영 시간** | Q01, Q02, Q03, Q04... | 의무실, 사지방, 체단실, PX 운영 시간 | 이용 가능한 일과 및 주말 시간 확인 |
| **신청 절차** | Q05, Q08, Q09, Q15... | 도서 대출, 휴가 서류, 이발실 예약, 상담 신청 | 서류 제출 및 사전 승인 절차 수반 |
| **담당 부서** | Q11, Q12, Q14... | 자격증 지원 담당과, 우편물 수령처 | 문의 및 처리를 담당하는 주관 부서 확인 |

---

## 2. 전입 첫 주 우선 질문 5선 (Top 5)
- 신병이 자대 적응 과정에서 가장 시급하게 알아야 할 질문 5개를 선별합니다.

1. **우선 질문 1 (Q01):** 의무실 일반 진료 시간 및 응급 접수 방법
   - **선정 사유:** 건강 및 응급 상황 대응과 직결되는 최우선 필수 복지 정보
2. **우선 질문 2 (Q04):** 복지매장(PX) 운영 시간 및 이용 규칙
   - **선정 사유:** 전입 초기 필수 세면도구 및 생필품 조달을 위한 필수 시설
3. **우선 질문 3 (Q02):** 사이버지식정보방(사지방) 이용 가능 시간
   - **선정 사유:** 일과 후 가족/지인 연락 및 자격증 준비를 위한 핵심 여가 시설
4. **우선 질문 4 (Q06):** 생활관 세탁기/건조기 이용 순번 및 시간
   - **선정 사유:** 매일 발생하는 피복 세탁 및 생활관 마찰 방지를 위한 기본 수칙
5. **우선 질문 5 (Q08):** 휴가/외출 신청 서류 제출처 및 기한
   - **선정 사유:** 첫 신병 위로휴가 준비 및 군 행정 절차 숙지 필수 항목

---

## 3. 현재 정보 탐색 방식과 불편한 점 (현장 문제 분석)
- **현재 탐색 경로:** 선임병 구두 질의(70%), 행정반 간부 직접 문의(20%), 게시판 육안 확인(10%)
- **주요 불편 사항:**
  1. *심리적 부담:* 전입 초기 맞선임이나 분대장에게 사소한 질문을 거듭하기 눈치 보임.
  2. *정보 왜곡 위험:* 구두 전파 특성상 선임의 기억 오류로 잘못된 시간이나 규정이 전달됨.
  3. *확인 지연:* 일과 중 담당 간부 부재 시 당일 확인이 불가능하여 행정 서류 제출 기한을 놓침.
- **AI 안내 도우미 도입 기대효과:**
  - 24시간 언제든 눈치 보지 않고 표준화된 규정을 3초 내 확인 가능
  - 공식 원문 데이터 기반 답변으로 정보 왜곡 원천 차단

### 📦 필수 학습자료 및 실습 팁
**미션 수행 필수 학습자료**

- **가상 한빛대대 병영생활 질문 카드 20선 (JSON/MD):** 신규 전입 장병들이 실제로 겪는 다빈도 질의 20건 원문 데이터셋
- **부대 생활 질문 분류 및 우선순위 선정 서식 (MD/XLSX):** 4대 카테고리(시설, 시간, 절차, 부서) 분류 및 Top 5 우선순위 도출 시트
- **[학습 가이드] 사용자 중심 문제 정의 및 페르소나 분석법:** 신병의 시각에서 병목 현상을 진단하고 AI 도입 우선순위를 분석하는 실무 가이드

**실습 팁:** 20개 질문을 나눌 때는 장병들이 '어디로 가야 하는가(시설)', '언제까지 가능한가(시간)', '어떤 서류를 내는가(절차)'를 기준으로 분리하면 명확합니다.

### 🔍 평가 기준 (PASS 조건)
#### 미션 판정
**평가 구분:** 필수

**핵심 평가 기준:** [필수] 문제 정의 — 자주 묻는 질문 20개를 체계적으로 분류하고 AI 안내 도우미 도입 필요성과 우선순위를 도출했는가? (PASS / FAIL)

#### 미션 PASS 조건

<li>20개 질문 카드 4대 영역 분류표 완성

</li><li>신병 전입 첫 주 우선 질문 5선 및 선정 사유 타당성 확보

</li><li>기존 정보 탐색 경로의 한계 및 AI 도우미 도입 기대효과 정리

</li><li>교육용 가상 데이터셋 활용 및 군사보안 수칙 엄수

</li>

#### 채점자 판정 메모
<blockquote>단순 질문 나열에 그치지 않고, 신병의 실제 애로사항(눈치, 정보 왜곡 등)을 정확히 진단하고 AI 도입 목표와 연결했는지 확인한다.

</blockquote>





In [ ]:
# [미션 01. 자주 묻는 부대 생활 정보 파악하기] 학생 실습 프롬프트 및 결과 기록란
mission_01_prompt = """
# [TODO: 미션 01] 질문 카드 20개를 4대 카테고리로 분류하고 전입 첫 주 우선순위 Top 5를 도출하는 프롬프트를 작성하세요.
# 힌트: load_dataset(1)로 20개 질문 카드를 확인한 후, AI에게 카테고리 분류 기준과 우선순위 선별을 지시합니다.
# (여기에 작성한 프롬프트를 입력하세요)
"""

mission_01_result = """
### 📋 [미션 01 실습 결과]
## 1. 부대 생활 질문 20개 분류표
| 카테고리 | 질문 번호 (ID) | 질문 내용 요약 | 분류 사유 |
| :--- | :--- | :--- | :--- |
| **시설 위치** | | | |
| **운영 시간** | | | |
| **신청 절차** | | | |
| **담당 부서** | | | |

## 2. 전입 첫 주 우선 질문 5선 (Top 5)
1. 
2. 
3. 
4. 
5. 

## 3. 현재 정보 탐색 방식의 한계와 AI 도우미 도입 효과
- **현장 문제점**: 
- **AI 도입 효과**:
"""

# 렌더러 실행 (결과를 시각적으로 확인합니다)
show_result("미션 01. 자주 묻는 부대 생활 정보 파악하기", mission_01_prompt, mission_01_result)

